<a href="https://colab.research.google.com/github/Sirichunchu23/hpc-fake-or-real-news-detection/blob/main/Fake_News_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!apt-get install gcc

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,743 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubunt

In [33]:
%%writefile hpc_classifier.c
#include <stdio.h>
#include <string.h>
#include <omp.h>

#define MAX_LINES 500
#define MAX_LEN 5000

char lines[MAX_LINES][MAX_LEN];

char *fake_words[] = {"shocking", "fake", "click", "rumor"};
char *real_words[] = {"official", "report", "government", "study"};

int count_keywords(char *text, char *keywords[], int size) {
    int score = 0;
    for (int i = 0; i < size; i++) {
        if (strstr(text, keywords[i])) score++;
    }
    return score;
}

int classify(char *text) {
    int fake_score = count_keywords(text, fake_words, 4);
    int real_score = count_keywords(text, real_words, 4);

    return (fake_score > real_score) ? 0 : 1;
}

int main() {
    FILE *file = fopen("news.csv", "r");

    if (!file) {
        printf("File not found!\n");
        return 1;
    }

    int n = 0;

    while (fgets(lines[n], MAX_LEN, file) != NULL && n < MAX_LINES) {
        n++;
    }

    fclose(file);

    omp_set_num_threads(4);

    int fake_total = 0, real_total = 0;

    #pragma omp parallel for reduction(+:fake_total, real_total)
    for (int i = 0; i < n; i++) {
        int result = classify(lines[i]);

        if (result == 0) fake_total++;
        else real_total++;

        printf("Thread %d → Line %d → %s\n",
               omp_get_thread_num(),
               i,
               result == 0 ? "🔴 Fake" : "🟢 Real");
    }

    printf("\nSummary:\nFake: %d\nReal: %d\n", fake_total, real_total);

    return 0;
}

Overwriting hpc_classifier.c


In [34]:
!gcc -fopenmp hpc_classifier.c -o hpc_classifier

In [35]:
!./hpc_classifier

Thread 3 → Line 375 → 🟢 Real
Thread 3 → Line 376 → 🟢 Real
Thread 3 → Line 377 → 🟢 Real
Thread 0 → Line 0 → 🟢 Real
Thread 1 → Line 125 → 🟢 Real
Thread 3 → Line 378 → 🟢 Real
Thread 3 → Line 379 → 🟢 Real
Thread 0 → Line 1 → 🔴 Fake
Thread 1 → Line 126 → 🟢 Real
Thread 1 → Line 127 → 🟢 Real
Thread 3 → Line 380 → 🟢 Real
Thread 2 → Line 250 → 🟢 Real
Thread 2 → Line 251 → 🟢 Real
Thread 3 → Line 381 → 🟢 Real
Thread 0 → Line 2 → 🟢 Real
Thread 1 → Line 128 → 🟢 Real
Thread 1 → Line 129 → 🔴 Fake
Thread 1 → Line 130 → 🟢 Real
Thread 1 → Line 131 → 🟢 Real
Thread 1 → Line 132 → 🟢 Real
Thread 1 → Line 133 → 🟢 Real
Thread 1 → Line 134 → 🟢 Real
Thread 1 → Line 135 → 🟢 Real
Thread 1 → Line 136 → 🟢 Real
Thread 1 → Line 137 → 🟢 Real
Thread 1 → Line 138 → 🟢 Real
Thread 1 → Line 139 → 🟢 Real
Thread 1 → Line 140 → 🟢 Real
Thread 1 → Line 141 → 🟢 Real
Thread 1 → Line 142 → 🟢 Real
Thread 1 → Line 143 → 🟢 Real
Thread 1 → Line 144 → 🟢 Real
Thread 1 → Line 145 → 🟢 Real
Thread 1 → Line 146 → 🟢 Real
Thread 1 → Line 147 